# Formula 1 Analysis Report

This notebook provides a comprehensive analysis of Formula 1 data spanning the entire history of the sport (1950–2026). It leverages the Gold-layer analytics tables built through the medallion architecture pipeline (Bronze → Silver → Gold) in the `f1_catalog` Unity Catalog.

In [0]:
%sql --name pipeline_overview
-- ============================================================
-- Pipeline Overview: Row counts across all layers
-- ============================================================
SELECT 'Bronze' AS layer, 'drivers' AS table_name, COUNT(*) AS row_count FROM f1_catalog.bronze.drivers
UNION ALL SELECT 'Bronze', 'constructors', COUNT(*) FROM f1_catalog.bronze.constructors
UNION ALL SELECT 'Bronze', 'circuits', COUNT(*) FROM f1_catalog.bronze.circuits
UNION ALL SELECT 'Bronze', 'races', COUNT(*) FROM f1_catalog.bronze.races
UNION ALL SELECT 'Bronze', 'results', COUNT(*) FROM f1_catalog.bronze.results
UNION ALL SELECT 'Bronze', 'qualifying', COUNT(*) FROM f1_catalog.bronze.qualifying
UNION ALL SELECT 'Bronze', 'pit_stops', COUNT(*) FROM f1_catalog.bronze.pit_stops
UNION ALL SELECT 'Bronze', 'lap_times', COUNT(*) FROM f1_catalog.bronze.lap_times
UNION ALL SELECT 'Bronze', 'driver_standings', COUNT(*) FROM f1_catalog.bronze.driver_standings
UNION ALL SELECT 'Bronze', 'constructor_standings', COUNT(*) FROM f1_catalog.bronze.constructor_standings
UNION ALL SELECT 'Bronze', 'seasons', COUNT(*) FROM f1_catalog.bronze.seasons
UNION ALL SELECT 'Bronze', 'status', COUNT(*) FROM f1_catalog.bronze.status
UNION ALL SELECT 'Silver', 'drivers', COUNT(*) FROM f1_catalog.silver.drivers
UNION ALL SELECT 'Silver', 'constructors', COUNT(*) FROM f1_catalog.silver.constructors
UNION ALL SELECT 'Silver', 'circuits', COUNT(*) FROM f1_catalog.silver.circuits
UNION ALL SELECT 'Silver', 'races', COUNT(*) FROM f1_catalog.silver.races
UNION ALL SELECT 'Silver', 'results', COUNT(*) FROM f1_catalog.silver.results
UNION ALL SELECT 'Silver', 'qualifying', COUNT(*) FROM f1_catalog.silver.qualifying
UNION ALL SELECT 'Silver', 'pit_stops', COUNT(*) FROM f1_catalog.silver.pit_stops
UNION ALL SELECT 'Silver', 'lap_times', COUNT(*) FROM f1_catalog.silver.lap_times
UNION ALL SELECT 'Silver', 'driver_standings', COUNT(*) FROM f1_catalog.silver.driver_standings
UNION ALL SELECT 'Silver', 'constructor_standings', COUNT(*) FROM f1_catalog.silver.constructor_standings
UNION ALL SELECT 'Silver', 'seasons', COUNT(*) FROM f1_catalog.silver.seasons
UNION ALL SELECT 'Silver', 'status', COUNT(*) FROM f1_catalog.silver.status
UNION ALL SELECT 'Gold', 'driver_performance', COUNT(*) FROM f1_catalog.gold.driver_performance
UNION ALL SELECT 'Gold', 'constructor_performance', COUNT(*) FROM f1_catalog.gold.constructor_performance
UNION ALL SELECT 'Gold', 'season_summary', COUNT(*) FROM f1_catalog.gold.season_summary
UNION ALL SELECT 'Gold', 'circuit_performance', COUNT(*) FROM f1_catalog.gold.circuit_performance
ORDER BY layer, table_name

In [0]:
%sql --name top_drivers
-- ============================================================
-- Top 20 Drivers by Race Wins
-- ============================================================
SELECT 
    driver_rank,
    driver_name,
    races,
    race_wins,
    podiums,
    total_points,
    ROUND(avg_finish_position, 2) AS avg_finish_pos,
    ROUND(avg_grid_position, 2) AS avg_grid_pos,
    ROUND((race_wins / races) * 100, 2) AS win_pct
FROM f1_catalog.gold.driver_performance
ORDER BY race_wins DESC, total_points DESC
LIMIT 20

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql --name top_constructors
-- ============================================================
-- Top 15 Constructors by Race Wins
-- ============================================================
SELECT 
    constructor_rank,
    constructor_name,
    races,
    race_wins,
    podiums,
    total_points,
    ROUND(avg_finish_position, 2) AS avg_finish_pos,
    ROUND((race_wins / races) * 100, 2) AS win_pct
FROM f1_catalog.gold.constructor_performance
ORDER BY race_wins DESC, total_points DESC
LIMIT 15

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql --name season_trends
-- ============================================================
-- Season Trends: Races, Drivers, Constructors per Season
-- ============================================================
SELECT 
    season,
    total_races,
    total_drivers,
    total_constructors,
    ROUND(total_points, 1) AS total_points,
    total_wins
FROM f1_catalog.gold.season_summary
ORDER BY season DESC

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql --name top_circuits
-- ============================================================
-- Top 15 Most Active Circuits by Race Count
-- ============================================================
SELECT 
    circuit_name,
    total_races,
    total_wins,
    ROUND(total_points, 1) AS total_points,
    ROUND(avg_finish_position, 2) AS avg_finish_pos
FROM f1_catalog.gold.circuit_performance
ORDER BY total_races DESC
LIMIT 15

In [0]:
%sql --name dq_summary
-- ============================================================
-- Data Quality Summary
-- ============================================================
SELECT 
    check_status,
    COUNT(*) AS check_count
FROM f1_catalog.gold.data_quality_audit
GROUP BY check_status
ORDER BY check_status

In [0]:
%sql --name dq_failures
-- ============================================================
-- Failed Data Quality Checks
-- ============================================================
SELECT 
    table_name,
    layer,
    check_name,
    failed_records,
    check_message
FROM f1_catalog.gold.data_quality_audit
WHERE check_status = 'FAIL'
ORDER BY table_name, check_name

In [0]:
%sql --name pipeline_watermark
-- ============================================================
-- Pipeline Watermark: Ingestion Status
-- ============================================================
SELECT 
    table_name,
    file_name,
    processed_timestamp
FROM f1_catalog.gold.pipeline_watermark
ORDER BY processed_timestamp DESC